In [2]:
from importlib.util import find_spec
import subprocess
import sys

missing = [name for name in ("numpy", "scipy") if find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

from pathlib import Path
from statistics import median
from time import perf_counter

import numpy as np
from scipy import sparse

rng = np.random.default_rng(42)
print("Библиотеки подключены ")

Библиотеки подключены 


In [3]:
# задание 6.1. Срезы одномерного массива
arr = np.arange(20)
print("Каждый третий с индекса 1 ", arr[1::3])
print("Индексы с 5 до 15 (без 15) в обратном порядке ", arr[5:15][::-1])
print("Последние пять в обратном порядке ", arr[:-6:-1])

Каждый третий с индекса 1  [ 1  4  7 10 13 16 19]
Индексы с 5 до 15 (без 15) в обратном порядке  [14 13 12 11 10  9  8  7  6  5]
Последние пять в обратном порядке  [19 18 17 16 15]


In [4]:
# задание 6.2. Срезы и диагональ матрицы
mat = rng.integers(0, 10, size=(6, 6))
print("Исходная матрица \n", mat)
print("Индексы строк 2–4 и столбцов 1–3 \n", mat[2:5, 1:4])
print("Главная диагональ ", np.diagonal(mat))
rows, cols = np.indices(mat.shape)
print("Элементы на клетках с чётной суммой индексов \n",
      mat[(rows + cols) % 2 == 0].reshape(6, 3))

Исходная матрица 
 [[0 7 6 4 4 8]
 [0 6 2 0 5 9]
 [7 7 7 7 5 1]
 [8 4 5 3 1 9]
 [7 6 4 8 5 4]
 [4 2 0 5 8 0]]
Индексы строк 2–4 и столбцов 1–3 
 [[7 7 7]
 [4 5 3]
 [6 4 8]]
Главная диагональ  [0 6 7 3 5 0]
Элементы на клетках с чётной суммой индексов 
 [[0 6 4]
 [6 0 9]
 [7 7 5]
 [4 3 9]
 [7 4 5]
 [2 5 0]]


In [5]:
# задание 6.3. Выбор по индексам
a = np.array([10, 20, 30, 40, 50, 60])
m = np.arange(12).reshape(3, 4)
print("Элементы массива ", a[[0, 2, 4]])
print("Три пары координат ", m[[0, 1, 2], [1, 3, 0]])
print("Блок 2×2 \n", m[np.ix_([0, 2], [1, 3])])

Элементы массива  [10 30 50]
Три пары координат  [1 7 8]
Блок 2×2 
 [[ 1  3]
 [ 9 11]]


In [6]:
# задание 6.4. Булевы маски
np.random.seed(42)
x = np.random.randn(1000)
print("Количество значений больше 1 ", np.count_nonzero(x > 1))
positive = x.copy()
positive[positive < 0] = 0
print("Замена отрицательных, первые 10 ", positive[:10])
changed = x.copy()
indices = np.flatnonzero(x[:-2] * x[2:] < 0) + 1
changed[indices] = 0
print("Исходные первые 10 ", x[:10])
print("После проверки соседей ", changed[:10])
print("Всего позиций с соседями разных знаков ", len(indices))

Количество значений больше 1  155
Замена отрицательных, первые 10  [0.49671415 0.         0.64768854 1.52302986 0.         0.
 1.57921282 0.76743473 0.         0.54256004]
Исходные первые 10  [ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337 -0.23413696
  1.57921282  0.76743473 -0.46947439  0.54256004]
После проверки соседей  [ 0.49671415 -0.1382643   0.          0.          0.          0.
  0.          0.         -0.46947439  0.54256004]
Всего позиций с соседями разных знаков  512


In [7]:
# задание 7.1. Создание big.dat порциями
path = Path("big.dat").resolve()
shape = (50_000, 200)
dtype = np.dtype("float32")
batch_size = 1000
data = np.memmap(path, dtype=dtype, mode="w+", shape=shape)
for start in range(0, shape[0], batch_size):
    stop = min(start + batch_size, shape[0])
    data[start:stop] = rng.random((stop - start, shape[1]), dtype=dtype)
data.flush()
del data
file_size = path.stat().st_size
assert file_size == shape[0] * shape[1] * dtype.itemsize
print("Файл ", path)
print(f"Размер  {file_size:,} байт")

Файл  /content/big.dat
Размер  40,000,000 байт


In [9]:
# задание 7.2. Минимум, максимум и среднее по батчам
data = np.memmap(path, dtype=dtype, mode="r", shape=shape)
minimum, maximum = np.inf, -np.inf
total = 0.0
for start in range(0, shape[0], batch_size):
    batch = data[start:start + batch_size]
    minimum = min(minimum, batch.min())
    maximum = max(maximum, batch.max())
    total += batch.sum(dtype=np.float64)
print("Минимум = ", minimum, "Максимум = ", maximum, "Среднее = ", total / data.size)
del batch, data

Минимум =  0.0 Максимум =  0.9999999 Среднее =  0.49993614274504183


In [8]:
# задание 7.3. Порядок хранения и обхода
def sum_rows(arr):
    sums = []
    for i in range(arr.shape[0]):
        total = 0.0
        for j in range(arr.shape[1]):
            total += arr[i, j]
        sums.append(total)
    return sums

def sum_cols(arr):
    sums = []
    for j in range(arr.shape[1]):
        total = 0.0
        for i in range(arr.shape[0]):
            total += arr[i, j]
        sums.append(total)
    return sums

c_mat = rng.random((3000, 3000))
f_mat = np.asfortranarray(c_mat)
for order, matrix in [("C", c_mat), ("F", f_mat)]:
    for direction, function, axis in [("строки", sum_rows, 1), ("столбцы", sum_cols, 0)]:
        start = perf_counter()
        sums = function(matrix)
        elapsed = perf_counter() - start
        np.testing.assert_allclose(sums, matrix.sum(axis=axis))
        print(f"Порядок {order}, {direction}: {elapsed:.3f} с")
del c_mat, f_mat, matrix

Порядок C, строки: 3.625 с
Порядок C, столбцы: 2.437 с
Порядок F, строки: 2.397 с
Порядок F, столбцы: 2.248 с


In [10]:
# задание 7.4. Вычисление выражения тремя способами
size = 10_000_000
a = rng.random(size)
b = rng.random(size)

expected_list = np.empty_like(a)
loop_seconds = 0.0
for start_idx in range(0, size, 100_000):
    stop_idx = min(start_idx + 100_000, size)
    a_list = a[start_idx:stop_idx].tolist()
    b_list = b[start_idx:stop_idx].tolist()
    block = []
    start = perf_counter()
    for ai, bi in zip(a_list, b_list):
        block.append((ai + bi) / (ai - bi + 1))
    loop_seconds += perf_counter() - start
    expected_list[start_idx:stop_idx] = block
print(f"Python - цикл со списками  {loop_seconds:.3f} с")
del a_list, b_list, block

start = perf_counter()
expected = (a + b) / (a - b + 1)
print(f"NumPy - одна строка  {perf_counter() - start:.3f} с")
np.testing.assert_allclose(expected_list, expected)
del expected_list

result, denominator = np.empty_like(a), np.empty_like(a)
start = perf_counter()
np.add(a, b, out=result)
np.subtract(a, b, out=denominator)
np.add(denominator, 1, out=denominator)
np.divide(result, denominator, out=result)
print(f"NumPy - готовые буферы out= {perf_counter() - start:.3f} с")
np.testing.assert_allclose(result, expected)
print("Все три результата совпадают ")
del a, b, result, denominator, expected

Python - цикл со списками  4.072 с
NumPy - одна строка  0.149 с
NumPy - готовые буферы out= 0.100 с
Все три результата совпадают 


In [12]:
# задание 8.1. Из COO в CSR с учётом дубликатов
row = rng.integers(0, 5000, 10_000).tolist()
col = rng.integers(0, 5000, 10_000).tolist()
values = rng.integers(1, 10, 10_000).tolist()
row[-1], col[-1] = row[0], col[0]
coo = sparse.coo_matrix((values, (row, col)), shape=(5000, 5000))
csr = coo.tocsr()
unique = len(set(zip(row, col)))
assert csr.nnz == unique
print("Записей COO ", coo.nnz, "Уникальных координат = ", unique, "nnz CSR ", csr.nnz)

Записей COO  10000 Уникальных координат =  9997 nnz CSR  9997


In [13]:
# задание 8.2. Память и время умножения на вектор
def multiply_time(matrix, vector):
    matrix @ vector
    times = []
    for _ in range(5):
        start = perf_counter()
        result = matrix @ vector
        times.append(perf_counter() - start)
    return median(times), result

def compare_matrices(n, density):
    csr = sparse.random(n, n, density=density, format="csr", dtype=np.float64,
                        random_state=rng, data_rvs=rng.random)
    dense = csr.toarray()
    vector = rng.random(n)
    dense_time, y_dense = multiply_time(dense, vector)
    csr_time, y_csr = multiply_time(csr, vector)
    np.testing.assert_allclose(y_dense, y_csr)
    csr_bytes = csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes
    return dense.nbytes, csr_bytes, dense_time, csr_time

dense_bytes, csr_bytes, dense_time, csr_time = compare_matrices(4000, 0.005)
for name, count, seconds in [("dense", dense_bytes, dense_time), ("CSR", csr_bytes, csr_time)]:
    print(f"{name}: {count:,} байт; A @ x  {seconds * 1000:.3f} мс")
print(f"В CSR данные занимают в {dense_bytes / csr_bytes:.1f} раза меньше памяти.")

dense: 128,000,000 байт; A @ x  9.666 мс
CSR: 976,004 байт; A @ x  0.203 мс
В CSR данные занимают в 131.1 раза меньше памяти.


In [14]:
# задание 8.3. Поиск порога по таблице замеров
densities = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
first = None
print("Плотность    dense, мс    CSR, мс    Быстрее")
for density in densities:
    _, _, dense_time, csr_time = compare_matrices(2000, density)
    winner = "CSR" if csr_time < dense_time else "dense"
    print(f"{density:9.3f}    {dense_time * 1000:9.3f}    {csr_time * 1000:7.3f}    {winner}")
    if first is None and csr_time >= dense_time:
        first = density
if first is None:
    print("CSR быстрее при всех проверенных плотностях ")
else:
    print(f"Первая проверенная плотность, при которой CSR перестаёт выигрывать  {first:.1%}.")

Плотность    dense, мс    CSR, мс    Быстрее
    0.001        2.559      0.036    CSR
    0.005        6.473      0.070    CSR
    0.010        6.387      0.091    CSR
    0.050        7.056      0.511    CSR
    0.100        3.951      0.834    CSR
    0.200        5.987      5.500    CSR
    0.500        3.997      7.772    dense
    1.000        3.430     10.684    dense
Первая проверенная плотность, при которой CSR перестаёт выигрывать  50.0%.
